<a href="https://colab.research.google.com/github/RADHESHYAM07/AI_powered_LinkedIN_Scraper/blob/main/AI_Powred_linkedIn_Scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 !pip install gradio #install gradio for webapp

In [ ]:
!pip install flair #flair for ai Skill detection


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.1/203.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import gradio as gr #Gradio to build webapp
import requests  # to fetch the web Pages
from bs4 import BeautifulSoup # this library is to parse HTML
import pandas as pd # to handle datatables
from datetime import datetime
import time # for adding delays in requests


# import flair tools for AI skill detection
from flair.models import SequenceTagger
from flair.data import Sentence

import threading # for smooth background Tasks
import os #to save File
import random  # for random Delays


# import retry tools to handle web error
from requests.adapters import HTTPAdapter
from urllib3.util.retry import RequestHistory

### Load AI MODEL

In [ ]:
# load Flair's Skill detection MOdel
flair_model =  SequenceTagger.load('kaliani/flair-ner-skill')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/169M [00:00<?, ?B/s]

2026-05-19 05:24:03,400 SequenceTagger predicts: Dictionary with 7 tags: O, S-SKILL, B-SKILL, E-SKILL, I-SKILL, <START>, <STOP>


Define Filter Mappings

In [ ]:
# Map experience level to linkedIn URL code
experience_level_mapping= {
    "Internship": "f_E=1",
    "Entry Level": "f_E=2",
    "Associate": "f_E=3",
    "Mid-Senior Level": "f_E=4"
}

# Map work type to LinkedIn URL code
work_type_mapping={
   "On-site": "f_WT=1",
   "Remote": "f_WT=2",
   "Hybrid": "f_WT=3"
}

# Map time filter to LinkedIn URL codes
time_filter_mapping = {
    "Past 24 hours": "f_TPR=r86400",
    "Past week": "f_TPR=r604800",
    "Past month": "f_TPR=r2592000"
}


In [ ]:
description = """Overview

As Data-logger, you will provide technical expertise and support to deliver exceptional service for data capture, management, and reporting throughout the entire lifecycle for wellsite drilling & workover operations. Using data reporting tools and techniques, you will apply your knowledge and understanding to ensure daily reporting of wellsite activity. This includes data capture from various service providers, client organization, real-time data sources and other data formats to ensure streamlined data capture via reporting tools. You will also monitor the reporting for mandatory parameters and maintain records to monitor daily reporting quality. You will receive training that expands your technical knowledge and hands-on skillset.

Responsibilities

Perform daily reporting of wellsite operations and be accountable for overall performance of timely data capture and reporting including configuration, loading, QC, monitoring and troubleshooting of operational activity for the asset / location
Ensure seamless data capture with reporting application including monitoring of mandatory reports, data sources, report formats including units, identifiers, meanings, normal value ranges to identify and fix data issues proactively.
Learn the reporting application, system installation and configuration, troubleshooting process and keep track of issues and resolution
Understand the role of different stakeholders and coordinate project data collection across service companies, client organization, real time data providers
Provide support to the on-site team, collaborate with the project team, client organization and remote operations support to ensure smooth and reliable function of application software and systems, including system upgrades, troubleshooting and maintenance activities
Monitor service quality, understand and utilize the Incident Escalation Procedures and pre-emptively inform supervisors when potential issues arise
Understand and Implement the Data Confidentiality Standards and IT Security Policies across all activities related to data capture, storage, transfer and archival
Maintain HSE Scorecards and actively contribute to HSE Safety Programs

Locations: Mehsana, Ahmedabad, Dehra Doon, East India, Assam etc.

Minimum Qualification And Experience

Minimum 3 years' Diploma in E&T / IT / Electronics / Computers/ EC / Electrical / Mechanical / Petroleum or Graduate in IT/BCA or equivalent/higher.
Minimum 2 years' experience of working in Oil & Gas/drilling company/organization
Good verbal and written communication skills in English.

Minimum Qualification And Experience

Minimum 3 years' Diploma in E&T / IT / Electronics / Computers/ EC / Electrical / Mechanical / Petroleum or Graduate in IT/BCA or equivalent/higher.
Minimum 2 years' experience of working in Oil & Gas upstream/drilling company/ etc.
Good verbal and written communication skills in English
"""

**Skill Detection Function**

In [ ]:
# define function to find skill in description

def get_skills(text):
  #turn text into flair model context
  sentense = Sentence(text)
  # use AI to detect skills
  flair_model.predict(sentense)
  # return List of Skills found
  return[entity.text for entity in sentense.get_spans('ner')]

get_skills(description)

['communication skills in', 'English']

**Scrapper Manager Class**

In [ ]:
#create a class to mange Scraping

class ScrapperManager:
  # setup when class starts
  def __init__(self):
    self.stop_event = threading.Event()
  # empty Table for data
    self.current_df = pd.DataFrame()
  # Lock to Avoid the dataMixUP
    self.lock = threading.Lock()

# Reset for New Scrape
def reset(self):
  # clear Stop Flag
  self.stop_event.clear()
  # clear Job table
  self.current_df = pd.DataFrame()


# Add One JOb to the Table
def add_job(self, job_data):
  # lock the data to keep safe
  with self.lock:
    # make job into tiny table
    new_df = pd.DataFrame([job_data])
    # add to main table
    self.current_df = pd.concat([self.current_df,new_df],ignore_index=True)

# Create Manage Instance

scraper_manger = ScrapperManager()


**Save to CSV Function**

In [ ]:
def save_csv(df, filename="jobs"):
    try:
  # make folder for files
     os.makedirs("saved_jobs", exists_ok=True)
     # get default name with Time stamp
     if not filename:
      filename = f"jobs_{int(time.time())}"
      # buld File path
      full_path = f"saved_jobs/{filename.csv}"
      # save table to csv
      df.to_csv(full_path, index=False)
      # confirmed save work
      return f"saved_to {full_path}"
    except Exception as e:
      return f"Save_error: {str(e)}"

Process job function

In [4]:
# define function to extract job function
def process_job(job, work_type, experience_level, position):
    try:
      # find the job title
      title_element = job.find('h3', class_='base-search-card__title')
      #find company name
      company_element = job.find('a', class_='hidden-nested-link')
      # find job location
      loc_element = job.find('span', class_='job-search-card__location')
      # find job link
      link_element = job.find('a', class_='base-card__full-link')
      # check all data exists
      if not all([title_element, company_element, loc_element, link_element]):
        return None
      # clean title text
      title = title_element.text.strip()
      #clean comapny text
      company = company_element.text.strip()
      # clean location text
      location = loc_element.text.strip()
      # clean link text
      link = link_element['href'].split('?')[0]

      # setup web sessions with retries
      session =  requests.Session()

      # set retry rules for error
      retries = RequestHistory(total = 3, backoff_factor = 0.1, status_forcelist = [429,500,502,503,504])

      # apply retries for session
      session.mount('https://',HTTPAdapter(max_retries=retries))


      #default if description fails
      desc = "Description Not Avilable"

      # empty Skills list
      skills = []

      # wait 2-5 seconds to avoid bans
      time.sleep(random.uniform(2,5))

      # fetch Job Page
      response = session.get(
          link,
          headers={
              # random brouser to seems like human
              'User-Agent': random.choice([
                  'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
                  'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.5 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.5'
              ])
          }
      )
    except Exception as e:
        pass